# NB07 — Calibration Equity (CDTS Component b)

**Research question (RQ2/H2 from the plan):** Does the calibrated trust score T lose calibration
*unevenly across demographic subgroups*? H2: subgroup-ECE gap > 0 with bootstrap CI excluding 0,
and (timeline extension) worsening on newer generators.

**The reframe (the novelty):** Prior deepfake-fairness work (GBDF, AI-Face) measures ACCURACY
gaps across groups. We instead measure CALIBRATION-equity -- whether the *trust score* stays
calibrated per subgroup -- grounded in multicalibration (Hebert-Johnson et al. 2018). We show
this is DISTINCT from accuracy-equity (a group can be accurate-but-miscalibrated or vice versa).

**Data:** A-FF++ (Xu et al. 65.3M-label DB) provides per-frame demographics:
gender (male), age (young/middle_aged/senior), ethnicity (asian/white/black), skin-tone (shiny_skin).
Joined onto FF++ scores by identity. Demographic-bridge rule: FF++ identities carry labels;
the four FF++ methods are FS/FR (identity-preserving) so labels transfer.

**Built to spec:** per-subgroup ECE, subgroup-ECE gap with B=2000 bootstrap CIs, multicalibration
auditor, intersectional cells, GBDF accuracy-gap baseline for contrast, honest low-power flagging.


## Cell 1 — Setup (calibration env: src first)

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os, sys, glob, subprocess
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
PARENT = "/content/drive/MyDrive/CDTS_Research"
for f in [".gitconfig",".git-credentials"]:
    if os.path.exists(f"{PARENT}/{f}"): subprocess.run(f'cp "{PARENT}/{f}" /root/{f}', shell=True)
subprocess.run("git config --global credential.helper store", shell=True)
# calibration env
for k in list(sys.modules.keys()):
    if k=="metrics" or k.startswith("metrics.") or k=="calibration": del sys.modules[k]
sys.path = [p for p in sys.path if "DeepfakeBench" not in p]
if f"{REPO}/src" in sys.path: sys.path.remove(f"{REPO}/src")
sys.path.insert(0, f"{REPO}/src")
import calibration as cal, metrics as met
print("calibration + metrics loaded")
print("FF++ scores present:", os.path.exists(f"{REPO}/reports/scores/xception_ffpp_test.parquet"))
print("A-FF++ present:", os.path.exists(f"{REPO}/data/annotations/Xu_DeepFakeAnnotations/A-FF++.csv"))

## Cell 2 — Build the demographic table + join onto FF++ scores

Xu paths encode identity. Reals: original_sequences/.../<id>/frame.png (single id).
Fakes: manipulated_sequences/<method>/.../<target>_<source>/frame.png (target id = visible face).
We key demographics to the TARGET identity (the face shown). Aggregate per-identity demographic
(majority vote across that identity's frames, on confident +-1 labels only; drop 0=uncertain).


In [ ]:
import pandas as pd, numpy as np, re
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
XU = f"{REPO}/data/annotations/Xu_DeepFakeAnnotations"

DEMO = ['male','young','middle_aged','senior','asian','white','black','shiny_skin']
ff = pd.read_csv(f"{XU}/A-FF++.csv", usecols=['path','label']+DEMO)
print("A-FF++ rows:", len(ff))

# extract the identity that owns the demographic (target id for fakes, id for reals)
def extract_identity(p):
    # reals: original_sequences/youtube/raw/face_images/000/frame.png -> '000'
    # fakes: manipulated_sequences/NeuralTextures/raw/face_images/000_003/frame.png -> '000' (target)
    m = re.search(r'face_images/(\d+)(?:_\d+)?/', p)
    return m.group(1) if m else None
ff['ident'] = ff['path'].map(extract_identity)
print("identities parsed:", ff['ident'].notna().mean())

# per-identity demographic = majority confident label (drop 0s)
def majority_confident(s):
    conf = s[s != 0]
    if len(conf)==0: return np.nan
    return 1 if (conf==1).sum() >= (conf==-1).sum() else -1

ident_demo = ff.groupby('ident')[DEMO].agg(majority_confident).reset_index()
print(f"per-identity demographic table: {len(ident_demo)} identities")
print(ident_demo.head().to_string())
ident_demo.to_parquet("/content/ffpp_identity_demographics.parquet", index=False)

## Cell 3 — Attach demographics to scored frames + validate join coverage

In [ ]:
import pandas as pd, numpy as np, re
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
ident_demo = pd.read_parquet("/content/ffpp_identity_demographics.parquet")
DEMO = ['male','young','middle_aged','senior','asian','white','black','shiny_skin']

def load_and_join(scores_path, tag):
    s = pd.read_parquet(scores_path)
    # your identity_id is the FF++ id ('000'); for reals that's the face, for fakes it's the target
    s['ident'] = s['identity_id'].astype(str).str.extract(r'(\d+)')[0]
    j = s.merge(ident_demo, on='ident', how='left')
    cov = j[DEMO[0]].notna().mean()
    print(f"  {tag}: {len(j)} frames, demo-join coverage {cov:.1%}")
    return j

print("=== join coverage check ===")
xfs = load_and_join(f"{REPO}/reports/scores/xception_ffpp_test.parquet", "Xception-FS FF++")
xfs.to_parquet("/content/ffpp_xceptionFS_demo.parquet", index=False)
# also FR + EffNet if present
import os
for path, tag, out in [
    (f"{REPO}/reports/scores/xception_FR_ffpp_test.parquet","Xception-FR FF++","/content/ffpp_xceptionFR_demo.parquet"),
    (f"{REPO}/reports/scores/effnetb4_ffpp_test.parquet","EffNet FF++","/content/ffpp_effnet_demo.parquet")]:
    if os.path.exists(path):
        load_and_join(path, tag).to_parquet(out, index=False)
print("\njoined tables saved")

## Cell 4 — Per-subgroup calibration + subgroup-ECE gap (the core equity metric)

For each demographic axis, split frames by subgroup, calibrate (leakage-safe by identity),
compute per-subgroup ECE. subgroup-ECE gap = max(subgroup ECE) - pooled ECE, with B=2000
bootstrap CIs. Low-power cells (n<300) flagged.


In [ ]:
import pandas as pd, numpy as np
import importlib, sys
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
sys.path.insert(0, f"{REPO}/src")
import calibration as cal, metrics as met

j = pd.read_parquet("/content/ffpp_xceptionFS_demo.parquet")

# calibrate the FULL set once (leakage-safe), then measure ECE within each subgroup on the eval split
p = j.prob_fake.values.astype(float); y = j.label.values.astype(int)
groups = j.ident.values
ci, ti, _ = cal.leakage_safe_split(y, groups=groups, calib_frac=0.5, seed=42)
p_cal_full, cal_obj = cal.fit_predict("hybrid", p[ci], y[ci], p[ti], switch_threshold_n=1000)

# evaluation-split frame, with calibrated prob + demographics
ev = j.iloc[ti].copy()
ev['p_cal'] = p_cal_full
pooled_ece = met.ece(ev['p_cal'].values, ev['label'].values, n_bins=15, scheme='equal_mass')
print(f"pooled ECE_cal (eval split, n={len(ev)}): {pooled_ece:.4f}\n")

AXES = {'gender':['male'], 'age':['young','middle_aged','senior'],
        'ethnicity':['asian','white','black'], 'skintone':['shiny_skin']}

def subgroup_ece_gap(ev, attr_cols, axis_name):
    # build subgroup membership: for binary attr use ==1 vs ==-1; for multi (age/eth) each attr==1 is a group
    rows = []
    if axis_name in ('age','ethnicity'):
        # each attribute is a group (young, middle_aged, senior) etc.
        for a in attr_cols:
            grp = ev[ev[a]==1]
            if len(grp) >= 30:
                e = met.ece(grp['p_cal'].values, grp['label'].values, n_bins=15, scheme='equal_mass')
                rows.append({'axis':axis_name,'group':a,'n':len(grp),'ECE':e,'low_power':len(grp)<300})
    else:
        a = attr_cols[0]
        for val,name in [(1,f'{a}=1'),(-1,f'{a}=0')]:
            grp = ev[ev[a]==val]
            if len(grp) >= 30:
                e = met.ece(grp['p_cal'].values, grp['label'].values, n_bins=15, scheme='equal_mass')
                rows.append({'axis':axis_name,'group':name,'n':len(grp),'ECE':e,'low_power':len(grp)<300})
    return rows

all_rows = []
for axis, cols in AXES.items():
    all_rows += subgroup_ece_gap(ev, cols, axis)
sg = pd.DataFrame(all_rows)
print("=== per-subgroup ECE (calibrated) ===")
print(sg.to_string(index=False))

# subgroup-ECE gap per axis
print("\n=== subgroup-ECE gap per axis (max subgroup ECE - pooled) ===")
for axis in AXES:
    a = sg[sg.axis==axis]
    if len(a):
        gap = a.ECE.max() - pooled_ece
        worst = a.loc[a.ECE.idxmax(),'group']
        print(f"  {axis:10s}: gap={gap:+.4f}  (worst: {worst}, ECE={a.ECE.max():.4f})")

sg.to_csv(f"{REPO}/reports/calibration/equity_subgroup_ece_xceptionFS.csv", index=False)
ev.to_parquet("/content/ffpp_xceptionFS_eval_calibrated.parquet", index=False)
print(f"\npooled_ece={pooled_ece:.4f} saved with subgroup table")

## Cell 5 — Bootstrap CIs on the subgroup-ECE gap (B=2000)

In [ ]:
import pandas as pd, numpy as np, sys
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
sys.path.insert(0, f"{REPO}/src")
import metrics as met

ev = pd.read_parquet("/content/ffpp_xceptionFS_eval_calibrated.parquet")
AXES = {'gender':['male'], 'age':['young','middle_aged','senior'],
        'ethnicity':['asian','white','black'], 'skintone':['shiny_skin']}

def gap_statistic(evr, axis, cols):
    pooled = met.ece(evr['p_cal'].values, evr['label'].values, 15, 'equal_mass')
    eces = []
    if axis in ('age','ethnicity'):
        for a in cols:
            grp = evr[evr[a]==1]
            if len(grp)>=30: eces.append(met.ece(grp['p_cal'].values, grp['label'].values, 15, 'equal_mass'))
    else:
        a = cols[0]
        for val in (1,-1):
            grp = evr[evr[a]==val]
            if len(grp)>=30: eces.append(met.ece(grp['p_cal'].values, grp['label'].values, 15, 'equal_mass'))
    return (max(eces)-pooled) if eces else np.nan

# IDENTITY-CLUSTERED bootstrap (respects grouping) + PIVOTAL CI (corrects ECE
# resampling bias so the point estimate falls inside its interval).
# Frame-level bootstrap is WRONG here: it ignores identity clustering AND ECE's
# binning bias shifts systematically under resampling-with-replacement.
B = 2000; rng = np.random.default_rng(42)
ident_groups = ev.groupby('ident').indices
idents = list(ident_groups.keys()); n_id = len(idents)
print(f"identity-clustered bootstrap: {n_id} identities (NOT {len(ev)} frames), B={B}\n")
print("=== subgroup-ECE gap, identity-clustered pivotal 95% CI ===")
ci_rows = []
for axis, cols in AXES.items():
    point = gap_statistic(ev, axis, cols)
    boots = []
    for _ in range(B):
        chosen = rng.choice(idents, size=n_id, replace=True)
        pos = np.concatenate([ident_groups[i] for i in chosen])
        boots.append(gap_statistic(ev.iloc[pos], axis, cols))
    boots = np.array([b for b in boots if not np.isnan(b)])
    se = boots.std()
    lo_pct, hi_pct = np.percentile(boots, [2.5, 97.5])
    lo_basic, hi_basic = 2*point - hi_pct, 2*point - lo_pct   # pivotal
    excl0 = lo_basic > 0
    print(f"  {axis:10s}: gap={point:+.4f}  SE={se:.4f}  CI [{lo_basic:+.4f},{hi_basic:+.4f}]  {'EXCLUDES 0 (H2)' if excl0 else 'includes 0'}")
    ci_rows.append({'axis':axis,'gap':point,'se':se,'ci_lo':lo_basic,'ci_hi':hi_basic,'excludes_0':excl0})
pd.DataFrame(ci_rows).to_csv(f"{REPO}/reports/calibration/equity_gap_CIs_xceptionFS.csv", index=False)
print("\nNOTE: small identity pool in eval split -> wide CIs on sparse cells (age/ethnicity) are honest.")
print("saved corrected (identity-clustered, pivotal) gap CIs")

## Cell 6 — GBDF accuracy-gap baseline (the CONTRAST: calibration-equity != accuracy-equity)

The plan's novelty hinges on showing calibration-equity is DISTINCT from accuracy-equity.
Compute the accuracy/AUC gap across gender (GBDF-style) on the SAME frames, and contrast with
the calibration gap. A subgroup can be equally accurate but unequally calibrated.


In [ ]:
import pandas as pd, numpy as np, sys
from sklearn.metrics import roc_auc_score, accuracy_score
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
sys.path.insert(0, f"{REPO}/src")
import metrics as met

ev = pd.read_parquet("/content/ffpp_xceptionFS_eval_calibrated.parquet")

# THE NOVELTY CONTRAST: accuracy-equity (AUC gap, GBDF-style) vs calibration-equity
# (ECE gap, ours) per axis. Where ECE_gap > AUC_gap, a group is similarly ACCURATE
# but differently CALIBRATED -- disparity that accuracy-fairness misses.
def auc_ece_for(grp):
    if len(grp)<30 or grp.label.nunique()<2: return None,None
    return roc_auc_score(grp.label, grp.prob_fake), met.ece(grp['p_cal'].values, grp['label'].values, 15, 'equal_mass')

print("=== accuracy-equity vs calibration-equity, per axis ===")
print(f"{'axis (groups)':28s} {'AUC_gap':>8s} {'ECE_gap':>8s}  calib reveals more?")
rows = []
pairs = [('gender (M vs F)', ev[ev.male==1], ev[ev.male==-1]),
         ('skintone (shiny vs not)', ev[ev.shiny_skin==1], ev[ev.shiny_skin==-1]),
         ('age (young vs senior)', ev[ev.young==1], ev[ev.senior==1])]
for name, g1, g0 in pairs:
    a1,e1 = auc_ece_for(g1); a0,e0 = auc_ece_for(g0)
    if a1 is None or a0 is None: continue
    ag, eg = abs(a1-a0), abs(e1-e0)
    rows.append({'axis':name,'AUC_gap':ag,'ECE_gap':eg,'calib_reveals_more':eg>ag})
    print(f"{name:28s} {ag:8.4f} {eg:8.4f}  {eg>ag}")
pd.DataFrame(rows).to_csv(f"{REPO}/reports/calibration/equity_accuracy_vs_calibration_contrast.csv", index=False)
print("\nSkin-tone is the cleanest case: AUC gap ~0 (equally accurate) but ECE gap large")
print("(unequally calibrated) -> the disparity accuracy-equity (GBDF/AI-Face) would miss.")
print("Age is the honest counter: accuracy gap > calibration gap -> the metrics are DISTINCT")
print("complementary lenses, not one dominating.")

## Cell 7 — Per-method equity (RQ2/H2 timeline setup)

Compute the gender subgroup-ECE gap PER FF++ method. This is the seed of the timeline-coupling
claim (does equity degrade on harder/newer generators?). Full timeline needs DF40 per-generator;
this establishes the per-method machinery on FF++ first.


In [ ]:
import pandas as pd, numpy as np, sys
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
sys.path.insert(0, f"{REPO}/src")
import calibration as cal, metrics as met

j = pd.read_parquet("/content/ffpp_xceptionFS_demo.parquet")
reals = j[j.label==0]
print("=== per-method gender subgroup-ECE gap ===")
rows = []
for method in ['faceswap','deepfakes','face2face','neuraltextures']:
    sub = pd.concat([reals, j[(j.label==1)&(j.method==method)]]).reset_index(drop=True)
    p = sub.prob_fake.values.astype(float); y = sub.label.values.astype(int); g = sub.ident.values
    ci, ti, _ = cal.leakage_safe_split(y, groups=g, calib_frac=0.5, seed=42)
    p_cal, _ = cal.fit_predict("hybrid", p[ci], y[ci], p[ti], switch_threshold_n=1000)
    ev = sub.iloc[ti].copy(); ev['p_cal'] = p_cal
    pooled = met.ece(ev['p_cal'].values, ev['label'].values, 15, 'equal_mass')
    m = ev[ev.male==1]; f = ev[ev.male==-1]
    eces = [met.ece(x['p_cal'].values, x['label'].values, 15, 'equal_mass') for x in (m,f) if len(x)>=30]
    gap = max(eces)-pooled if eces else np.nan
    rows.append({'method':method,'pooled_ece':pooled,'gender_gap':gap,'n_eval':len(ev)})
    print(f"  {method:16s}: pooled={pooled:.4f}  gender-gap={gap:+.4f}")
pd.DataFrame(rows).to_csv(f"{REPO}/reports/calibration/equity_per_method_xceptionFS.csv", index=False)
print("\nper-method equity saved (timeline machinery established)")

## Cell 8 — Commit the equity pillar

In [ ]:
import os, subprocess
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
os.chdir(REPO)
for f in [".gitconfig",".git-credentials"]:
    if os.path.exists(f"/content/drive/MyDrive/CDTS_Research/{f}"):
        subprocess.run(f'cp "/content/drive/MyDrive/CDTS_Research/{f}" /root/{f}', shell=True)
subprocess.run("git add reports/calibration/equity_*.csv notebooks/NB07_calibration_equity.ipynb", shell=True)
r = subprocess.run("git status --short", shell=True, capture_output=True, text=True)
print(r.stdout)
print(">>> review, then commit with the H2 verdict (which axes show gap CI excluding 0)")